# router_final — training, gate post-hoc, deployment A3, evaluasi (Kaggle T4)

Jalankan berurutan di kernel baru. Satu seed per sesi Kaggle (batas 12 jam): training + gate + ekspor + evaluasi.
Kode = folder `router_final/` di branch `final-arsitektur`, di atas **Ultralytics stok 8.3.240** (bukan fork di root repo).

In [ ]:
# ===== 0. Kode dipin =====
REPO_URL = "https://github.com/maugi1547/thesis-Difficulty-Aware-Router"
CODE_COMMIT = "ISI_HASH_COMMIT"          # hash commit branch final-arsitektur yang dipakai (wajib, bukan nama branch)
SEED = 42                                # satu seed per sesi: 42, 7, 2026
CONFIG = "configs/kitti.yaml"
RUN_REPRO_CHECK = True                   # 2 × 2 epoch, bandingkan bobot (±30 menit). Wajib sekali per lingkungan baru.
RUN_ABLATION = SEED == 42                # ablasi pasca-training (±1 jam): cukup seed 42 + VisDrone
# Baseline stok, TIDAK dilatih ulang. last.pt = aturan sama dengan router (Tahap 4/4B lama memakai best.pt;
# angka baseline tidak sebanding dengan tabel lama). Sensitivitas: ganti ke best.pt dan jalankan ulang sel 7 saja.
BASELINES = {
    "configs/kitti.yaml": dict(
        vanilla="/kaggle/input/models/agungmaugi/yolov8n-kitti/pytorch/default/1/Skenario1_YOLOv8n_Standar/weights/last.pt",
        p2static="/kaggle/input/models/agungmaugi/yolov8n-p2-statis-kitti/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/last.pt"),
    "configs/visdrone.yaml": dict(
        vanilla="/kaggle/input/models/agungmaugi/yolov8n-visdrone/pytorch/default/1/content/thesis-Difficulty-Aware-Router/Tesis_VisDrone/Skenario1_YOLOv8n_Standar/weights/last.pt",
        p2static="/kaggle/input/models/agungmaugi/yolov8n-p2-visdrone/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/last.pt"),
}
assert CODE_COMMIT != "ISI_HASH_COMMIT", "Isi CODE_COMMIT"

import os, subprocess
REPO = "/kaggle/working/thesis-Difficulty-Aware-Router"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--all"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", CODE_COMMIT], check=True)
RF = f"{REPO}/router_final"
os.chdir(RF)                              # PENTING: jangan di root repo (folder ultralytics/ fork akan menutupi paket pip)
!pip install -q --no-deps ultralytics==8.3.240
!pip install -q onnx onnxscript onnxruntime pycocotools tensorrt
print(subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True))

In [ ]:
# ===== 1. Uji unit (CPU, <1 menit): ultralytics stok, graf, loss, gate, B2, ONNX A3 =====
!python -m pytest -q -p no:cacheprovider tests

In [ ]:
# ===== 2. Uji reproduksi GPU: resep sama dua kali -> bobot harus identik bit per bit =====
if RUN_REPRO_CHECK:
    !python scripts/check_reproducibility.py --config {CONFIG} --seed {SEED} --override epochs=2 project=/kaggle/working/repro

In [ ]:
# ===== 3. Training detektor (100 epoch) =====
!python scripts/train.py --config {CONFIG} --seed {SEED}
import yaml, glob
cfg = yaml.safe_load(open(CONFIG))
RUN = f"{cfg['train']['project']}/{cfg['name']}_seed{SEED}"
print(RUN); print(open(f"{RUN}/state_hash.json").read())
# last.pt vs best.pt (best = fitness Branch A saja) di VAL, A dan B + bootstrap; laporan saja, last.pt tetap dipakai
!python scripts/check_checkpoint.py --config {CONFIG} --run {RUN} --out {RUN}/cek_checkpoint


In [ ]:
# ===== 4. Gate post-hoc (train) + thr0 (val) =====
!python scripts/fit_gate.py --config {CONFIG} --weights {RUN}/weights/last.pt --out {RUN}/gate

In [ ]:
# ===== 5. Ekspor A3 (ONNX If FP16) + verifikasi ORT + engine TensorRT =====
!python scripts/export.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/deploy --trt

In [ ]:
# ===== 6. Evaluasi test: TensorRT (sistem deploy) + PyTorch FP32 (pemeriksaan silang) =====
!python scripts/evaluate.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/eval_test_trt --backend trt --engine {RUN}/deploy/a3_fp16.engine
!python scripts/evaluate.py --config {CONFIG} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json --out {RUN}/eval_test_pytorch --backend pytorch

In [ ]:
# ===== 7. Evaluasi final terkunci + latensi ujung-ke-ujung (H1–H7, bootstrap, baseline, heuristik, kaskade) =====
# ±1,5 jam T4: cache train/val/test + 2 build engine + pass akurasi + latensi komponen + latensi ujung-ke-ujung + bootstrap
B = BASELINES[CONFIG]
!python scripts/suite.py run --config {CONFIG} --seed {SEED} --weights {RUN}/weights/last.pt --gate {RUN}/gate/gate.json \
    --vanilla "{B['vanilla']}" --p2static "{B['p2static']}" --out /kaggle/working/suite --zip /kaggle/working


In [ ]:
# ===== 8. Ablasi pasca-training (G1 target, G3 fitur, G4 classifier, G5 label, B anggaran) =====
if RUN_ABLATION:
    !python scripts/ablate.py --config {CONFIG} --weights {RUN}/weights/last.pt --out {RUN}/ablasi


In [ ]:
# ===== 9. Arsip (tanpa engine & ONNX besar) =====
import shutil, pathlib
keep = pathlib.Path(f"/kaggle/working/arsip_{cfg['name']}_seed{SEED}")
shutil.copytree(RUN, keep, ignore=shutil.ignore_patterns("*.engine", "*.onnx", "epoch*.pt"), dirs_exist_ok=True)
shutil.make_archive(str(keep), "zip", keep)
print(keep.with_suffix(".zip"))
print(sorted(str(p) for p in pathlib.Path("/kaggle/working").glob("suite_*.zip")))   # hasil + intermediate evaluasi final


## Ringkasan gabungan (sesi terakhir; CPU cukup)
Unggah semua `suite_<model>.zip` (bukan `_intermediate`) sebagai satu dataset Kaggle, lalu jalankan sel ini saja (setelah sel 0).

In [ ]:
# ===== 10. Gabung 4 model: tabel kriteria, pooling KITTI 3 seed, H3 + Holm, Pareto =====
MERGE_GLOB = "/kaggle/input/**/suite_*"     # sesuaikan dengan nama dataset hasil
!python scripts/suite.py merge --out /kaggle/working/suite_gabung --glob "{MERGE_GLOB}" \
    --models kitti_seed42 kitti_seed7 kitti_seed2026 visdrone_seed42
